# Task 1: Traffic Sign Classification with PyTorch

This notebook builds and trains a convolutional neural network for the GTSRB traffic sign dataset using a Hugging Face dataset source.

Network type: **Convolutional Neural Network (CNN)** for **multi-class image classification**.

## Training Structure

Because the dataset is fairly large, the workflow is organized as:

1. Load the dataset from Hugging Face
2. Split training into train and validation subsets
3. Apply light augmentation to training images only
4. Train in mini-batches with a `DataLoader`
5. Track validation loss and accuracy each epoch
6. Save the best checkpoint
7. Evaluate once on the test set

In [ ]:
import copy
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/Users/shamik/Documents/paiml/csci4170-paiml/hw5/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Community mirror of GTSRB hosted on Hugging Face.
# If this identifier changes, replace it with the current GTSRB dataset id.
HF_DATASET_ID = "ilee0022/GTSRB"

IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.1
NUM_WORKERS = 2

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_traffic_sign_cnn.pt"


## Load Dataset From Hugging Face

The dataset should provide image samples and labels. The training split is further divided into train and validation subsets.

In [ ]:
dataset = load_dataset(HF_DATASET_ID)
dataset

In [ ]:
dataset["train"][0]

In [ ]:
train_valid = dataset["train"].train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds_hf = train_valid["train"]
val_ds_hf = train_valid["test"]
test_ds_hf = dataset["test"]

sample_keys = list(train_ds_hf.features.keys())

if "image" in sample_keys:
    image_key = "image"
elif "Path" in sample_keys:
    image_key = "Path"
else:
    raise KeyError(f"Could not find an image column. Available columns: {sample_keys}")

if "label" in sample_keys:
    label_key = "label"
elif "Label" in sample_keys:
    label_key = "Label"
elif "ClassId" in sample_keys:
    label_key = "ClassId"
else:
    raise KeyError(f"Could not find a label column. Available columns: {sample_keys}")

num_classes = len(set(train_ds_hf[label_key]))
print(f"Train samples: {len(train_ds_hf)}")
print(f"Validation samples: {len(val_ds_hf)}")
print(f"Test samples: {len(test_ds_hf)}")
print(f"Image column: {image_key}")
print(f"Label column: {label_key}")
print(f"Number of classes: {num_classes}")

## Transforms

Training images get light augmentation. Validation and test images only get deterministic preprocessing.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
class HFGTSRBDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, image_key="image", label_key="label"):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.image_key = image_key
        self.label_key = label_key

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]

        image = sample[self.image_key]
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        else:
            image = image.convert("RGB")

        label = int(sample[self.label_key])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = HFGTSRBDataset(train_ds_hf, transform=train_transform, image_key=image_key, label_key=label_key)
val_dataset = HFGTSRBDataset(val_ds_hf, transform=eval_transform, image_key=image_key, label_key=label_key)
test_dataset = HFGTSRBDataset(test_ds_hf, transform=eval_transform, image_key=image_key, label_key=label_key)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

len(train_loader), len(val_loader), len(test_loader)

## Model

The convolutional base uses three repeated blocks:

- Block 1: `Conv -> ReLU -> Conv -> ReLU -> MaxPool`
- Block 2: `Conv -> ReLU -> Conv -> ReLU -> MaxPool`
- Block 3: `Conv -> ReLU -> Conv -> ReLU -> MaxPool`

The classifier uses:

- `Flatten -> Linear -> ReLU -> Dropout -> Linear`

The output layer predicts one of 43 traffic sign classes.

In [ ]:
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = TrafficSignCNN(num_classes=num_classes).to(device)
model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)


## Training and Evaluation Functions

In [ ]:
def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_training):
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_weights = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, BEST_MODEL_PATH)

    model.load_state_dict(best_weights)
    return model, history


In [ ]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS
)

print(f"Best model saved to: {BEST_MODEL_PATH}")

## Final Test Evaluation

The test set is used only after training and validation are complete.

In [ ]:
def predict_dataset(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)

In [ ]:
test_labels, test_preds = predict_dataset(model, test_loader)
test_accuracy = accuracy_score(test_labels, test_preds)

print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(test_labels, test_preds))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm

## Notes

- If GPU memory is limited, reduce `BATCH_SIZE`.
- If training is unstable, lower the learning rate.
- If the Hugging Face dataset schema uses different column names, update `image_key` and `label_key` in `HFGTSRBDataset`.
- `CrossEntropyLoss` expects raw logits, so the model should not include a `Softmax` layer.